# 01 · Baseline brain tumour MRI classifier

**Goal:** train a model that sorts a brain MRI slice into one of four classes (glioma, meningioma, no tumour, pituitary), and evaluate it honestly.

**How to use this notebook**
1. In Colab: `Runtime → Change runtime type → T4 GPU`, then Save.
2. Run cells top to bottom with `Shift + Enter`. Read the text above each cell first.
3. The only cell you normally edit is **Settings** (section 2). To try a Vision Transformer, change `MODEL_NAME` there and run everything again.

A full run with the default settings takes roughly 10–20 minutes on Colab's free T4 GPU.

## 1 · Install and import

Colab already has PyTorch. We add two small libraries:
- **timm**: a library of pretrained image models (ResNets, ViTs and hundreds more) with one consistent interface.
- **kagglehub**: downloads Kaggle datasets with one line.

Lines starting with `!` are shell commands, not Python.

In [ ]:
!pip install -q timm kagglehub

In [ ]:
import os, json, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")

## 2 · Settings

Everything you are likely to change lives here.

- `MODEL_NAME`: `"resnet18"` (a classic CNN, fast) or `"vit_small_patch16_224"` (a Vision Transformer). Both start from weights pretrained on ImageNet, which is called **transfer learning**: the model already knows generic visual features (edges, textures, shapes), and we only adapt it to MRI.
- `EPOCHS`: one epoch = one pass through all training images.
- `LR`: learning rate, the step size of gradient descent. Pretrained models need small steps so they don't forget what they know.
- `SEED`: fixes the randomness so that a re-run gives (nearly) the same numbers. Change it to see how much results vary between runs.

In [ ]:
MODEL_NAME = "resnet18"          # or "vit_small_patch16_224"
EPOCHS     = 8
BATCH_SIZE = 32
LR         = 3e-4
VAL_FRAC   = 0.15                # fraction of the Training folder held out for validation
SEED       = 42
IMG_SIZE   = 224

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

## 3 · Download the data

We use the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) by Masoud Nickparvar (about 7,000 images). It is public, so no Kaggle login is normally needed. If the download asks for credentials, see the Troubleshooting section of the tutorial doc.

The dataset comes already split into a `Training` folder and a `Testing` folder, each with one sub-folder per class. That layout is exactly what PyTorch's `ImageFolder` expects: **the folder name is the label**.

In [ ]:
import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
print("Downloaded to:", DATA_ROOT)

In [ ]:
# Find the Training and Testing folders wherever they sit inside the download
def find_dir(root, names):
    for p in sorted(Path(root).rglob("*")):
        if p.is_dir() and p.name.lower() in names:
            return p
    raise FileNotFoundError(f"No folder named one of {names} under {root}")

TRAIN_DIR = find_dir(DATA_ROOT, {"training", "train"})
TEST_DIR  = find_dir(DATA_ROOT, {"testing", "test"})
print("Train folder:", TRAIN_DIR)
print("Test folder: ", TEST_DIR)

## 4 · Look at the data before modelling

Always do this first. Check: how many images per class (is it balanced?), and what the images actually look like (different sizes, orientations, contrasts, borders?).

In [ ]:
full_train = datasets.ImageFolder(TRAIN_DIR)
test_raw   = datasets.ImageFolder(TEST_DIR)
CLASSES = full_train.classes
assert CLASSES == test_raw.classes, "Train and test class folders differ"
print("Classes:", CLASSES)

def counts(ds):
    c = np.bincount(ds.targets, minlength=len(ds.classes))
    return dict(zip(ds.classes, c.tolist()))

print("Training folder:", counts(full_train), "total", len(full_train))
print("Testing folder: ", counts(test_raw),   "total", len(test_raw))

In [ ]:
# A grid of random examples with their labels and original sizes
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax, idx in zip(axes.flat, random.sample(range(len(full_train)), 18)):
    img, label = full_train[idx]
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{CLASSES[label]}\n{img.size[0]}x{img.size[1]}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Train / validation / test splits

Three sets, three jobs:
- **Train**: the model learns from these.
- **Validation**: we check progress on these after each epoch and keep the best model. Carved out of the `Training` folder, stratified so each class keeps its proportion.
- **Test**: the `Testing` folder. We touch it **once**, at the very end. If you tune settings by looking at test scores, the test score stops being an honest estimate.

⚠️ **A known weakness to remember for later:** this dataset has no patient IDs, and slices from the same patient (and near-duplicate images) can end up on both sides of a split. That inflates scores. Project 3 (the leakage check) investigates exactly this, so write down your numbers here as the "before" figures.

In [ ]:
idx_all = np.arange(len(full_train))
train_idx, val_idx = train_test_split(
    idx_all, test_size=VAL_FRAC, stratify=full_train.targets, random_state=SEED)
print(f"train: {len(train_idx)}   val: {len(val_idx)}   test: {len(test_raw)}")

## 6 · Build the model

`timm.create_model(..., pretrained=True, num_classes=4)` loads ImageNet weights and replaces the final layer (the "head") with a fresh one that outputs 4 numbers, one score (logit) per class.

We also ask timm which image size and normalisation the pretrained weights expect, so the pre-processing matches what the model saw during pretraining.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(device)
data_cfg = timm.data.resolve_model_data_config(model)
MEAN, STD = data_cfg["mean"], data_cfg["std"]
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M parameters | mean={MEAN} std={STD}")

## 7 · Pre-processing and data augmentation

Every image is converted to 3-channel RGB (pretrained models expect 3 channels), resized to 224×224, turned into a tensor and normalised.

During **training only**, we also apply small random changes (crops, flips, rotations). This **data augmentation** shows the model slightly different versions of each image every epoch, which reduces overfitting. Validation and test images are never augmented, so their scores are reproducible.

Is a left–right flip medically harmless here? For tumour *type* it is a reasonable assumption, but it's worth thinking about which augmentations are safe for which clinical questions.

In [ ]:
to_rgb = transforms.Lambda(lambda im: im.convert("RGB"))

train_tf = transforms.Compose([
    to_rgb,
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    to_rgb,
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Two views of the same folder (one augmented, one not), then pick indices for each split
train_ds = Subset(datasets.ImageFolder(TRAIN_DIR, transform=train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(TRAIN_DIR, transform=eval_tf),  val_idx)
test_ds  = datasets.ImageFolder(TEST_DIR, transform=eval_tf)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device == "cuda"))
train_loader = DataLoader(train_ds, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **loader_kw)

## 8 · The training loop

This is the heart of deep learning, and worth reading line by line. For each batch:

1. **Forward pass**: `logits = model(x)` gives a score per class for each image.
2. **Loss**: cross-entropy. With softmax probabilities $p_k = e^{z_k}/\sum_j e^{z_j}$ and true class $y$, the loss is $-\log p_y$. Its gradient with respect to the logits is simply $p - \text{onehot}(y)$, a nice one to derive by hand.
3. **Backward pass**: `loss.backward()` computes the gradient of the loss with respect to every parameter (backpropagation = the chain rule, applied efficiently).
4. **Step**: `optimizer.step()` moves the parameters a small step downhill. AdamW is gradient descent with per-parameter adaptive step sizes and weight decay.

`model.train()` / `model.eval()` switch layers like dropout and batch-norm between training and inference behaviour. `torch.no_grad()` skips gradient bookkeeping when we only want predictions.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)                 # 1. forward
        loss = criterion(logits, y)       # 2. loss
        loss.backward()                   # 3. backward
        optimizer.step()                  # 4. step
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for x, y in loader:
        all_logits.append(model(x.to(device)).cpu())
        all_y.append(y)
    return torch.cat(all_logits), torch.cat(all_y)

def evaluate(model, loader):
    logits, y = predict(model, loader)
    loss = criterion(logits, y).item()
    acc = (logits.argmax(1) == y).float().mean().item()
    return loss, acc

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, ckpt_path = 0.0, RESULTS_DIR / f"best_{MODEL_NAME}.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    va_loss, va_acc = evaluate(model, val_loader)
    scheduler.step()
    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
        history[k].append(v)
    flag = ""
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), ckpt_path)
        flag = "  <- best so far, saved"
    print(f"epoch {epoch:2d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time()-t0:.0f}s{flag}")

## 9 · Learning curves

What to look for:
- Both losses falling: the model is learning.
- Training loss keeps falling but validation loss rises: **overfitting** (memorising the training images).
- Validation accuracy suspiciously close to 100% very early: be sceptical, and think about duplicates/leakage.

In [ ]:
ep = range(1, EPOCHS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, history["train_loss"], "o-", label="train"); a1.plot(ep, history["val_loss"], "o-", label="val")
a1.set_title("Loss"); a1.set_xlabel("epoch"); a1.legend()
a2.plot(ep, history["train_acc"], "o-", label="train"); a2.plot(ep, history["val_acc"], "o-", label="val")
a2.set_title("Accuracy"); a2.set_xlabel("epoch"); a2.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / f"curves_{MODEL_NAME}.png", dpi=120); plt.show()

## 10 · Final evaluation on the test set (run once)

We reload the best checkpoint (chosen by validation accuracy, never by test) and evaluate on the untouched test folder.

Accuracy alone hides a lot, so we also report per-class **precision** (of the images predicted as class k, how many really are k), **recall** (of the real class-k images, how many we caught) and the **confusion matrix**. In a clinical setting, a missed tumour (low recall) usually costs far more than a false alarm. That asymmetry is what "cost-sensitive learning" in the PhD listing refers to.

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_logits, test_y = predict(model, test_loader)
test_pred = test_logits.argmax(1)
test_acc = (test_pred == test_y).float().mean().item()
print(f"TEST accuracy: {test_acc:.4f}\n")
print(classification_report(test_y, test_pred, target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(test_y, test_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap="Blues", values_format="d")
plt.title(f"{MODEL_NAME}: test confusion matrix")
plt.tight_layout(); plt.savefig(RESULTS_DIR / f"confusion_{MODEL_NAME}.png", dpi=120); plt.show()

## 11 · Look at the mistakes

The most useful 5 minutes of any project. Are the errors random, or is there a pattern (a class pair the model confuses, a particular scan orientation, low-contrast images, odd crops)? Note down what you see: these observations are good material for your write-up and research proposal.

In [ ]:
probs = test_logits.softmax(1)
wrong = (test_pred != test_y).nonzero().flatten().tolist()
print(f"{len(wrong)} misclassified out of {len(test_y)}")
show = wrong[:12]
if show:
    cols = 6; rows = (len(show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 2.6 * rows), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, i in zip(axes.flat, show):
        img, _ = test_raw[i]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"true: {CLASSES[test_y[i]]}\npred: {CLASSES[test_pred[i]]} ({probs[i].max():.2f})", fontsize=8)
    plt.tight_layout(); plt.show()

## 12 · Log the result

Copy the printed line into the results table in your repository's `README.md`. Keeping a log of every run (including bad ones) is a research habit worth starting now.

In [ ]:
summary = {
    "model": MODEL_NAME, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE, "seed": SEED,
    "best_val_acc": round(best_val_acc, 4), "test_acc": round(test_acc, 4),
}
with open(RESULTS_DIR / f"summary_{MODEL_NAME}_seed{SEED}.json", "w") as f:
    json.dump(summary, f, indent=2)
print(summary)
print(f"\nREADME row:\n| {time.strftime('%Y-%m-%d')} | {MODEL_NAME} | {EPOCHS} | {LR} | {SEED} | {best_val_acc:.3f} | {test_acc:.3f} | |")

## 13 · What next?

1. **Switch to a Vision Transformer:** set `MODEL_NAME = "vit_small_patch16_224"` in section 2, then `Runtime → Run all`. Compare accuracy, the confusion matrix and the training time with ResNet. If the ViT's loss jumps around, try `LR = 1e-4`.
2. **Repeat with 2–3 different `SEED` values** and report mean ± spread. A single run can be lucky.
3. **Save this notebook to GitHub:** `File → Save a copy in GitHub`.
4. Next notebooks: `02_gradcam.ipynb` (where is the model looking?) and `03_leakage_check.ipynb` (how many test images are near-duplicates of training images?).